Copyright (c) Microsoft Corporation.
Licensed under the MIT License.

## 1. Before You Begin

**Which MAI-Image model should we reach for when we need images at volume?** MAI-Image-2.5-Flash is the production-efficiency variant of MAI-Image-2.5. It uses the same generation and editing endpoints, and it is tuned for lower latency and lower cost per image. In this notebook we generate batches, measure latency and cost, run requests concurrently, and then put Flash next to the base model on the same prompts so we can judge the trade-off ourselves.

> **Legacy model version.** A deployment of MAI-Image-2.5-Flash version 2026-06-02 is a legacy model version, and it retires on 2026-11-14 at 7:00 PM local time. When the version retires, inferencing returns error responses and all deployments of this version stop being supported, including every request in this notebook. To avoid disruption, edit the deployment to move to a supported version if a new one is available. MAI-Image-2.6-Flash, in the table below, is the newer generation.

### How Flash compares

| | MAI-Image-2.5 | MAI-Image-2.5-Flash (this notebook) | MAI-Image-2.6-Flash |
|---|---|---|---|
| What it is | The base model of the 2.5 family | The speed- and cost-tuned variant of 2.5 | The newer Flash model, from the 2.6 generation |
| Tuned for | Maximum fidelity | Latency, throughput, and cost per image | Speed and efficiency, with the 2.6 features |
| Generate and edit | Yes | Yes, with the same endpoints and size limits | Yes, plus multi-reference editing (up to five images) and web grounding |
| Output controls | 768 px minimum per side, 1,048,576 pixels maximum | The same | Adaptive aspect ratios and 1.5K output |
| Released | 2026-06-02 | 2026-06-02 | 2026-09-04, in public preview |
| Pricing | Per image | Per image, lower than the base model | Per million tokens (text in, image in, image out) |

The 2.6 details come from Microsoft's [announcement](https://techcommunity.microsoft.com/blog/azure-ai-foundry-blog/mai-image-2-6-and-mai-image-2-6-flash-quality-and-speed-at-production-scale/4550970). Because 2.6 is priced per token and 2.5 per image, we can't compare their costs without token counts from our own requests.

### Which one do we pick?

- **MAI-Image-2.5-Flash** when we generate many images and latency or cost per image drives the decision. This is the model this notebook measures.
- **MAI-Image-2.5** when fidelity matters more than speed. Section 6 puts both models on the same prompts, so we can see what Flash gives up, if anything, for our own subject matter.
- **MAI-Image-2.6-Flash** when we want the newer generation, multi-reference editing, or web grounding, and we can work with a public preview. The announcement compares 2.6 with 2.5 on quality and compares 2.6-Flash with other vendors' models on speed, so we would measure it against 2.5-Flash on our own prompts. Section 10 suggests how.

**Prerequisites**

1. A Microsoft Foundry project with MAI-Image-2.5-Flash deployed.
   See [models/quickstart/](https://github.com/nitya/model-releases/blob/e2d0f011d2f735ae018cfbd61a43393e0b992380/models/quickstart/README.md) for first-time setup.
2. The environment variables listed in section 2. A fourth, for the base MAI-Image-2.5 deployment, is optional and only used by the comparison in section 6.
3. `requests` installed: `pip install requests`

**API constraints** ([source: Learn docs](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-mai-image?tabs=python))

| Rule | Value |
|---|---|
| Minimum dimension (each) | 768 px |
| Maximum total pixels | 1,048,576 |
| Output format | PNG only |
| Auth header | `api-key` |


## 2. Set up your environment

The cell below verifies the three required variables and derives `BASE_ENDPOINT`.
It also reads `AZURE_MAI_IMAGE_25_DEPLOYMENT` if it's set. That is the base MAI-Image-2.5
deployment, which the comparison in section 6 uses. Without it, the notebook still runs
and only that comparison is skipped.

It also imports `time` and `concurrent.futures`, used in later sections for
throughput measurement and concurrent requests.


In [ ]:
%pip install requests python-dotenv --quiet

In [ ]:
from dotenv import load_dotenv
load_dotenv()

# Your .env file needs these variables:
#   MICROSOFT_FOUNDRY_ENDPOINT=https://<resource>.services.ai.azure.com/api/projects/<project>
#   MICROSOFT_FOUNDRY_API_KEY=<your key>
#   AZURE_MAI_IMAGE_25_FLASH_DEPLOYMENT=MAI-Image-2.5-Flash
# Optional, for the comparison in section 5:
#   AZURE_MAI_IMAGE_25_DEPLOYMENT=MAI-Image-2.5


In [ ]:
import os, base64, time
import requests
import concurrent.futures
from pathlib import Path
from urllib.parse import urlparse
from IPython.display import HTML, Image as IPyImage, display

REQUIRED = {
    "MICROSOFT_FOUNDRY_ENDPOINT":             "Foundry project endpoint",
    "MICROSOFT_FOUNDRY_API_KEY":               "AIServices API key",
    "AZURE_MAI_IMAGE_25_FLASH_DEPLOYMENT":     "MAI-Image-2.5-Flash deployment name",
}
missing = [k for k in REQUIRED if not os.environ.get(k)]
if missing:
    raise EnvironmentError(f"Set these env vars before continuing: {missing}")

parsed        = urlparse(os.environ["MICROSOFT_FOUNDRY_ENDPOINT"])
BASE_ENDPOINT = f"{parsed.scheme}://{parsed.netloc}"
API_KEY       = os.environ["MICROSOFT_FOUNDRY_API_KEY"]
DEPLOYMENT    = os.environ["AZURE_MAI_IMAGE_25_FLASH_DEPLOYMENT"]
# Optional: the base MAI-Image-2.5 deployment, used by the comparison in section 6.
BASE_DEPLOYMENT = os.environ.get("AZURE_MAI_IMAGE_25_DEPLOYMENT")

print(f"Base endpoint : {BASE_ENDPOINT}")
print(f"Deployment    : {DEPLOYMENT}")
print(f"Comparison    : {BASE_DEPLOYMENT or 'skipped (set AZURE_MAI_IMAGE_25_DEPLOYMENT to enable)'}")
print("Environment check passed.")


## 3. Configure the client

Flash supports both `/mai/v1/images/generations` and `/mai/v1/images/edits`. The shared helpers preserve service error details, avoid replacing output files, decode returned PNG images, and display results inline.

In [ ]:
import binascii
from html import escape

GEN_URL = f"{BASE_ENDPOINT}/mai/v1/images/generations"

OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

REQUEST_TIMEOUT = 300


def generate_image(prompt: str, width: int = 1024, height: int = 1024, deployment: str | None = None) -> str:
    """POST to /mai/v1/images/generations; return base64 PNG string.

    Uses the Flash deployment unless `deployment` names another one.
    """
    assert width >= 768 and height >= 768, "Each dimension must be ≥ 768 px"
    assert width * height <= 1_048_576,    "width × height must be ≤ 1,048,576"
    resp = requests.post(
        GEN_URL,
        headers={"Content-Type": "application/json", "api-key": API_KEY},
        json={"model": deployment or DEPLOYMENT, "prompt": prompt, "width": width, "height": height},
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()["data"][0]["b64_json"]


def show(b64: str, save_as: str | None = None) -> None:
    """Decode base64 PNG, optionally save, and display inline."""
    raw = base64.b64decode(b64)
    if save_as:
        out = OUT_DIR / save_as
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_bytes(raw)
        print(f"Saved → {out}")
    display(IPyImage(data=raw))


def show_compare(panels: list[tuple[str, bytes, str]], width: int = 420) -> None:
    """Show (label, png_bytes, caption) panels side by side at the same width."""
    # A fixed width keeps differences down to the model, not the scaling.
    figures = "".join(
        '<figure style="margin:0;text-align:center">'
        f'<img src="data:image/png;base64,{base64.b64encode(raw).decode()}" style="width:{width}px;height:auto">'
        f"<figcaption><b>{label}</b><br>{caption}</figcaption></figure>"
        for label, raw, caption in panels
    )
    display(HTML(f'<div style="display:flex;gap:16px;justify-content:center">{figures}</div>'))


def show_grid(tiles: list[tuple[str, bytes, str]], columns: int = 4, tile_px: int = 240) -> None:
    """Show (label, png_bytes, caption) tiles in a grid with `columns` per row."""
    # Square tiles with object-fit: contain keep mixed image shapes aligned without cropping.
    cells = "".join(
        '<figure style="margin:0;text-align:center">'
        f'<img src="data:image/png;base64,{base64.b64encode(raw).decode()}" '
        'style="width:100%;aspect-ratio:1/1;object-fit:contain;background:#f2f2f2;border-radius:6px">'
        f'<figcaption style="font-size:12px"><b>{escape(label)}</b><br>{escape(caption)}</figcaption></figure>'
        for label, raw, caption in tiles
    )
    display(HTML(
        f'<div style="display:grid;grid-template-columns:repeat({columns},minmax(0,1fr));'
        f'gap:12px;max-width:{columns * tile_px}px;margin:auto">{cells}</div>'
    ))


def versioned_path(path: str) -> str:
    candidate = Path(path)
    if not candidate.exists():
        return str(candidate)
    for version in range(2, 10_000):
        next_path = candidate.with_stem(f'{candidate.stem}_v{version}')
        if not next_path.exists():
            return str(next_path)
    raise RuntimeError('Could not find an available output path.')


def post_image_api(route: str, *, json=None, data=None, files=None) -> dict:
    started = time.perf_counter()
    print(f'Model: {DEPLOYMENT}')
    response = requests.post(f'{BASE_ENDPOINT}/mai/v1/images/{route}', headers={'api-key': API_KEY}, json=json, data=data, files=files, timeout=REQUEST_TIMEOUT)
    if not response.ok:
        raise requests.HTTPError(f'{response.status_code} {response.reason}: {response.text}', response=response)
    print(f'API runtime: {time.perf_counter() - started:.2f}s')
    return response.json()


def save_images(result: dict, output_path: str) -> list[str]:
    images = [item['b64_json'] for item in result.get('data', []) if item.get('b64_json')]
    if not images:
        raise ValueError(f'Unexpected response format: {result}')
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    paths = []
    for index, encoded in enumerate(images, start=1):
        path = output if len(images) == 1 else output.with_stem(f'{output.stem}_{index}')
        try:
            path.write_bytes(base64.b64decode(encoded))
        except (binascii.Error, ValueError) as exc:
            raise ValueError(f'Could not decode image {index}.') from exc
        paths.append(str(path.resolve()))
        print(f'Saved: {paths[-1]}')
    return paths


## 4. Generate a batch of images

**How much does the wording of a prompt change what we get?** A batch is a list of prompts processed one after another. This one is eight prompts that differ on purpose:

- **Style:** photorealistic, stylized 3D animation, flat vector illustration, watercolor, and pixel art
- **Camera position:** eye level, close-up, side view, wide shot, top-down, low angle, and macro
- **Color and subject size:** a cobalt mug, a coral cottage, a water droplet, a skyscraper
- **Image shape:** square, landscape, and portrait, all within the pixel budget from section 1

The grid shows all eight, four per row. Each caption gives the image size and how long that request took. Notice how the words that name the style and the camera position ("photorealistic", "top-down", "low-angle") often change the result more than the object does.

Running them one at a time also gives us a baseline before we parallelise in section 7.


In [ ]:
BATCH_ITEMS = [
    # (label, prompt, width, height). Sizes stay within the pixel budget from section 1.
    ("Photorealistic · eye level",
     "Photorealistic studio photograph of a cobalt-blue ceramic coffee mug on a white background, "
     "soft-box lighting, sharp focus, eye-level camera", 1024, 1024),
    ("3D animated · close-up",
     "Close-up still from a stylized 3D animated film: a tiny cheerful red robot watering a giant "
     "yellow sunflower, soft golden light, expressive eyes", 1024, 768),
    ("Flat illustration · side view",
     "Flat vector illustration, side view, of a mint-green bicycle leaning against a sunny yellow wall, "
     "bold shapes, limited palette, no gradients", 768, 1024),
    ("Watercolor · wide shot",
     "Loose watercolor painting, wide shot, of a lavender field at dusk with a small white farmhouse "
     "and a pink-orange sky", 1024, 768),
    ("Top-down photo · flat lay",
     "Top-down flat-lay photograph of a breakfast table: a glass of orange juice, a croissant, a bowl of "
     "blueberries and a white plate on a pale oak surface, natural window light", 1024, 1024),
    ("Low angle · architecture",
     "Dramatic low-angle photograph looking straight up at a tall glass skyscraper against a deep orange "
     "sunset sky, wide-angle lens, converging lines", 768, 1024),
    ("Macro · extreme close-up",
     "Extreme macro photograph of a single water droplet on a bright green leaf, shallow depth of field, "
     "soft backlight, crisp texture", 1024, 1024),
    ("Isometric pixel art",
     "Isometric pixel-art illustration of a coral-red cottage on a small floating island with a waterfall, "
     "bright blue sky, crisp pixels", 1024, 768),
]


def run_item(item: tuple[str, str, int, int]) -> tuple[bytes, float]:
    """Generate one batch item; return the PNG bytes and the request time in seconds."""
    _, prompt, width, height = item
    started = time.perf_counter()
    raw = base64.b64decode(generate_image(prompt, width=width, height=height))
    return raw, time.perf_counter() - started


t_start = time.monotonic()
tiles = []

for i, item in enumerate(BATCH_ITEMS):
    label, _, width, height = item
    raw, seconds = run_item(item)
    path = OUT_DIR / "sequential_batch" / f"batch-seq-{i:02d}.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(raw)
    tiles.append((label, raw, f"{width}×{height} · {seconds:.1f}s"))

elapsed = time.monotonic() - t_start
show_grid(tiles, columns=4)   # 8 images: 4 per row, 2 rows
print(f"\nSequential batch: {len(BATCH_ITEMS)} images in {elapsed:.1f}s "
      f"({elapsed/len(BATCH_ITEMS):.1f}s / image)")


## 5. Measure throughput and estimate cost

Before parallelising, record the single-request latency distribution.
This gives a baseline to compare against the concurrent run in section 7.

The cell below times one image for each of the first four prompts from section 4, all at 1024×1024 so the timings are comparable. Different subjects and styles can take different amounts of time, so the spread between the minimum and maximum is worth a look.
At current Flash pricing, estimate cost as `n_images × price_per_image`.


In [ ]:
# USD per image. Update these if pricing changes; check the model page for current rates.
PRICE_PER_IMAGE = 0.04        # MAI-Image-2.5-Flash
BASE_PRICE_PER_IMAGE = 0.05   # MAI-Image-2.5, used in the comparison in section 6

latencies = []
# The first four batch prompts, all at 1024×1024 so the timings are comparable.
sample_prompts = [item[1] for item in BATCH_ITEMS[:4]]

for prompt in sample_prompts:
    t0 = time.monotonic()
    b64 = generate_image(prompt, width=1024, height=1024)
    latencies.append(time.monotonic() - t0)

avg_latency = sum(latencies) / len(latencies)
print(f"Samples        : {len(latencies)}")
print(f"Avg latency    : {avg_latency:.2f}s / image")
print(f"Min / Max      : {min(latencies):.2f}s / {max(latencies):.2f}s")
print()

for n in (10, 100, 1_000, 10_000):
    est_cost  = n * PRICE_PER_IMAGE
    est_time  = n * avg_latency
    print(f"  {n:>6} images → ${est_cost:>8.2f}  |  sequential ETA ≈ {est_time/60:,.1f} min")


## 6. Compare Flash with MAI-Image-2.5

**Is Flash's speed and cost saving worth any difference in the image?** We send the same two prompts to both models, one at a time, and show each pair side by side with its latency and estimated cost.

- **Product shot** is the kind of clean, simple image a catalog pipeline produces. This is where we'd expect Flash to hold up.
- **Fine detail** asks for skin texture and individual rope fibres, which is where a fidelity gap is most likely to show.

Look at both pairs before you read the numbers. Then ask: for *our* images, would we notice the difference at the size they'll be shown?

Latency here is wall-clock time for one request, so it includes network time and will vary from run to run. Cost uses the per-image prices set in the section 5 cell, so run that cell first. This section also needs the base model deployed. If `AZURE_MAI_IMAGE_25_DEPLOYMENT` isn't set, the cell skips itself.


In [ ]:
COMPARE_PROMPTS = [
    ("Product shot",
     "Studio product shot of a cobalt-blue ceramic coffee mug on a white background, "
     "soft-box lighting, sharp focus, minimal drop shadow"),
    ("Fine detail",
     "Close-up of a weathered fisherman's hands tying a rope knot, natural window light, "
     "visible skin texture and individual rope fibres"),
]

COMPARE_MODELS = [
    # (label, deployment, USD per image)
    ("MAI-Image-2.5-Flash", DEPLOYMENT, PRICE_PER_IMAGE),
    ("MAI-Image-2.5", BASE_DEPLOYMENT, BASE_PRICE_PER_IMAGE),
]


def timed_generate(prompt: str, deployment: str) -> tuple[bytes, float]:
    started = time.perf_counter()
    b64 = generate_image(prompt, width=1024, height=1024, deployment=deployment)
    return base64.b64decode(b64), time.perf_counter() - started


if not BASE_DEPLOYMENT:
    print("Set AZURE_MAI_IMAGE_25_DEPLOYMENT in your .env file to compare with MAI-Image-2.5. Skipping this section.")
else:
    results = []   # (prompt title, model label, seconds, cost)
    # Requests run one at a time so concurrency doesn't distort the latency numbers.
    for title, prompt in COMPARE_PROMPTS:
        print(f"{title}: {prompt}")
        panels = []
        for label, deployment, price in COMPARE_MODELS:
            raw, seconds = timed_generate(prompt, deployment)
            out = OUT_DIR / "compare" / f"{title.lower().replace(' ', '-')}-{label.lower()}.png"
            out.parent.mkdir(parents=True, exist_ok=True)
            out.write_bytes(raw)
            panels.append((label, raw, f"{seconds:.1f}s · ~${price:.2f}"))
            results.append((title, label, seconds, price))
        show_compare(panels)

    print(f"\n{'Prompt':<14}{'Model':<22}{'Latency':>9}{'Est. cost':>11}")
    for title, label, seconds, price in results:
        print(f"{title:<14}{label:<22}{seconds:>8.1f}s{'$' + format(price, '.2f'):>11}")

    print()
    for title, _ in COMPARE_PROMPTS:
        flash = next(r for r in results if r[0] == title and r[1] == "MAI-Image-2.5-Flash")
        base = next(r for r in results if r[0] == title and r[1] == "MAI-Image-2.5")
        faster = (base[2] - flash[2]) / base[2] * 100
        cheaper = (base[3] - flash[3]) / base[3] * 100
        print(f"{title}: Flash was {abs(faster):.0f}% {'faster' if faster >= 0 else 'slower'} "
              f"and {cheaper:.0f}% cheaper per image than MAI-Image-2.5.")


## 7. Run concurrent requests

`concurrent.futures.ThreadPoolExecutor` issues requests in parallel.
Each worker is a blocking HTTP call, so threads (not processes) work here.

The cell below runs the same eight prompts from section 4 and shows the same grid. Compare the total time with the sequential run to see what concurrency buys us.

**Choose `max_workers` carefully:**
- Start with 4–8 workers to avoid rate-limit errors
- Check your Foundry quota page for the TPM / RPM limits on your deployment
- Increase gradually and re-run this cell to find your practical ceiling


In [ ]:
MAX_WORKERS = 4   # increase after confirming your quota allows it


def _generate(args):
    idx, item = args
    raw, seconds = run_item(item)
    return idx, raw, seconds


t_start = time.monotonic()

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    results = list(executor.map(_generate, enumerate(BATCH_ITEMS)))

total_elapsed = time.monotonic() - t_start

tiles = []
for idx, raw, seconds in results:
    label, _, width, height = BATCH_ITEMS[idx]
    path = OUT_DIR / "concurrent_batch" / f"batch-par-{idx:02d}.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(raw)
    tiles.append((label, raw, f"{width}×{height} · {seconds:.1f}s"))

show_grid(tiles, columns=4)   # same 8 prompts: 4 per row, 2 rows

images_per_sec = len(BATCH_ITEMS) / total_elapsed
print(f"\nConcurrent batch ({MAX_WORKERS} workers): "
      f"{len(BATCH_ITEMS)} images in {total_elapsed:.1f}s "
      f"({images_per_sec:.2f} images/s)")


## 8. Create multiple background options whilst maintaining original feature

Keeping the clothing item photorealistic preserves accurate color, texture, condition, and construction details. Editing only the background lets a seller reuse one truthful product image across multiple visual settings without the time and cost of repeated photo shoots.

Use the edits endpoint to place one clothing photo in six marketplace-ready settings while preserving the item. Add a JPEG or PNG at `images/blue-dress.jpg`, or update `CLOTHING_IMAGE` in the next cell. Results are versioned under `output/` and shown in a grid, three per row.


In [ ]:
cell_started = time.perf_counter()
print(f'Model: {DEPLOYMENT}')

import mimetypes

RESALE_LISTING_BACKGROUNDS = [
    ('paris_apartment', 'a bright Parisian apartment with pale walls, subtle molding, and warm oak flooring'),
    ('fashion_boutique', 'an elegant independent fashion boutique with neutral decor and soft window light'),
    ('minimal_studio', 'a clean warm-white photography studio with a seamless backdrop and soft diffused lighting'),
    ('luxury_wardrobe', 'a refined walk-in wardrobe with muted finishes and an uncluttered editorial look'),
    ('hotel_suite', 'a sophisticated boutique hotel suite with tasteful neutral furnishings and natural daylight'),
    ('stone_townhouse', 'a chic European townhouse interior with pale stone details and understated styling'),
]

def create_resale_listing_background_variations(image_path: str, output_dir: str = 'output') -> list[str]:
    source = Path(image_path)
    if not source.is_file():
        raise FileNotFoundError(f'Clothing image not found: {source}')
    mime_type = mimetypes.guess_type(source.name)[0]
    if mime_type not in {'image/jpeg', 'image/png'}:
        raise ValueError('The clothing image must be a JPEG or PNG image.')

    def create_variation(item: tuple[int, tuple[str, str]]) -> list[str]:
        index, (name, background) = item
        prompt = ('Edit only the background of this secondhand clothing photo. Place the clothing item in ' + background + '. ' 'Keep the item and every foreground detail unchanged: shape, cut, proportions, color, pattern, fabric texture, seams, labels, embellishments, condition, and fit. ' 'Do not redesign, retouch, reshape, recolor, or add anything to the item. Match the original perspective and use physically realistic lighting and shadows. ' 'Create an authentic professional product photograph suitable for an online clothing resale listing. No text, logos, borders, or watermarks.')
        output_path = versioned_path(str(Path(output_dir) / f'{source.stem}_resale_listing_backgrounds' / f'{index:02d}_{name}.png'))
        print(f'Creating variation {index}/{len(RESALE_LISTING_BACKGROUNDS)}: {name}')
        with source.open('rb') as image_file:
            result = post_image_api('edits', data={'model': DEPLOYMENT, 'prompt': prompt}, files={'image': (source.name, image_file, mime_type)})
        return save_images(result, output_path)

    indexed_backgrounds = list(enumerate(RESALE_LISTING_BACKGROUNDS, start=1))
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        variation_paths = executor.map(create_variation, indexed_backgrounds)
    return [path for paths in variation_paths for path in paths]

CLOTHING_IMAGE = 'images/blue-dress.jpg'  # Replace with your clothing image.
resale_listing_paths = create_resale_listing_background_variations(CLOTHING_IMAGE)

# executor.map keeps the input order, so the paths line up with the backgrounds.
tiles = [
    (name.replace('_', ' ').title(), Path(path).read_bytes(), Path(path).name)
    for (name, _), path in zip(RESALE_LISTING_BACKGROUNDS, resale_listing_paths)
]
show_grid(tiles, columns=3, tile_px=300)   # 6 images: 3 per row, 2 rows
print(f'Total runtime: {time.perf_counter() - cell_started:.2f}s')

## 9. Generate visual mood boards

Generate six art directions for **The AI Engineer's desk** while keeping the composition rules fixed. This makes the visual treatment easier to compare across materials, lighting, and accent colors. The first cell generates and saves the boards, and the second shows them in a grid, three per row, so we can compare them at a glance.


In [ ]:
cell_started = time.perf_counter()
print(f'Model: {DEPLOYMENT}')

AI_ENGINEER_DESK_DIRECTIONS = [
    ('precision_lab', 'precision research lab, white ceramic, brushed aluminum, cool daylight, cyan accents'),
    ('warm_startup', 'warm creative startup studio, walnut desk, soft amber light, plants, coral and teal accents'),
    ('midnight_ops', 'midnight AI operations workspace, charcoal surfaces, focused monitor glow, electric blue accents'),
    ('calm_minimal', 'calm minimalist workspace, pale ash wood, matte white tools, diffused morning light, sage accents'),
    ('analog_digital', 'analog-meets-digital workshop, notebooks, technical sketches, mechanical keyboard, moss and rust tones'),
    ('future_luxury', 'quiet futuristic luxury office, smoked glass, black metal, sculptural lighting, silver accents'),
]

def generate_ai_engineer_desk_mood_boards(output_dir: str = 'output/ai_engineers_desk_mood_boards') -> list[str]:
    def generate_mood_board(item: tuple[int, tuple[str, str]]) -> list[str]:
        index, (name, direction) = item
        prompt = (
            "Create a sophisticated visual mood board for the concept 'The AI Engineer's desk'. "
            f'Art direction: {direction}. Compose a cohesive editorial collage with one hero workspace image and smaller supporting frames showing tactile materials, desk objects, computing hardware, abstract neural-network diagrams, lighting details, and a restrained row of color swatches. '
            'Professional design presentation, photorealistic product photography, precise grid, generous spacing, coherent lighting, premium art-direction quality, square composition. No people, brand logos, watermarks, captions, labels, or legible text.'
        )
        print(f'Creating mood board {index}/{len(AI_ENGINEER_DESK_DIRECTIONS)}: {name}')
        result = post_image_api('generations', json={'model': DEPLOYMENT, 'prompt': prompt, 'width': 1024, 'height': 1024})
        output_path = versioned_path(str(Path(output_dir) / f'{index:02d}_{name}.png'))
        return save_images(result, output_path)

    indexed_directions = list(enumerate(AI_ENGINEER_DESK_DIRECTIONS, start=1))
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        mood_board_paths = executor.map(generate_mood_board, indexed_directions)
    return [path for paths in mood_board_paths for path in paths]

mood_board_paths = generate_ai_engineer_desk_mood_boards()
print(f'Total runtime: {time.perf_counter() - cell_started:.2f}s')

In [ ]:
# Directions and paths share the same order, so they line up one to one.
tiles = [
    (name.replace('_', ' ').title(), Path(path).read_bytes(), Path(path).name)
    for (name, _), path in zip(AI_ENGINEER_DESK_DIRECTIONS, mood_board_paths)
]
show_grid(tiles, columns=3, tile_px=300)   # 6 mood boards: 3 per row, 2 rows

## 10. Your Turn to Explore

1. In section 4, pick one prompt and rewrite only its style and camera words, for example "photorealistic, eye level" to "claymation, from above". What changes, and what stays the same?
2. Add your own prompts to `COMPARE_PROMPTS` in section 6, using the kind of image your pipeline really produces. Does Flash still hold up?
3. Add MAI-Image-2.6-Flash as a third entry in `COMPARE_MODELS`. Deploy it with `./scripts/3-deploy-model.sh --list MAI-Image-2.6`, then compare its latency and image quality. It is priced per token, so check the response and your Foundry usage metrics for token counts before you estimate its cost.


In [ ]:
# Your experiments here


## 11. Summary

You used MAI-Image-2.5-Flash to:

- Generate a batch of eight prompts that vary style, camera position, color, subject size, and image shape, and show them in a grid
- Run that batch sequentially and concurrently, and measure latency, estimated cost, and throughput
- Compare its images, latency, and cost with MAI-Image-2.5 on the same prompts
- Create six options of backgrounds for a source image
- Generate six coordinated visual mood boards

Choose Flash for image pipelines where latency, throughput, and cost matter more than maximum fidelity, and check that claim against your own prompts in section 6. For the newer generation, see MAI-Image-2.6-Flash in "Before You Begin".

**Next steps**

- [MAI-Image-2.5 capsule](../mai-image-generate-and-edit/) - baseline model with image editing
- [MAI-Image-2.5-Pro capsule](../mai-image-pro-portraits-and-text/) - portrait quality and text rendering
- [Image generation primer](https://github.com/nitya/model-releases/blob/e2d0f011d2f735ae018cfbd61a43393e0b992380/docs/primers/image-generation.md)


## 12. References

- [Deploy and use MAI image models in Microsoft Foundry](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-mai-image?tabs=python) - official generations and edits API guidance
- [Build 2026 MAI keynote transcript](https://microsoft.ai/news/microsoft-build-2026-mai-keynote-transcript/) - introduction to the MAI model family
- [MAI-Image-2.5 model page](https://microsoft.ai/models/mai-image-2-5/) - model overview and capability comparison
- [MAI-Image-2.6 and MAI-Image-2.6-Flash](https://techcommunity.microsoft.com/blog/azure-ai-foundry-blog/mai-image-2-6-and-mai-image-2-6-flash-quality-and-speed-at-production-scale/4550970) - the newer generation: features, pricing, and when to choose each
